In [ ]:
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
import os
import pickle

DATA = "FB15k"
ds_path = f"/home/cc/phd/KGEmbeddings/data/{DATA}/"

triples = pd.read_csv(ds_path + "merged.txt", sep="\t", header=None, names=["head", "relation", "tail"])
entities = pd.read_csv(ds_path + "entities.txt", sep="\t", header=None, names=["id", "entity"])
relations = pd.read_csv(ds_path + "relations.txt", sep="\t", header=None, names=["id", "relation"])

entity2id = dict(zip(entities["entity"], entities["id"]))
relation2id = dict(zip(relations["relation"], relations["id"]))

triples["head_id"] = triples["head"].map(entity2id)
triples["relation_id"] = triples["relation"].map(relation2id)
triples["tail_id"] = triples["tail"].map(entity2id)

ds = triples[["head_id", "relation_id", "tail_id"]]

missing_entities = ds[ds.isnull().any(axis=1)]

if not missing_entities.empty:
    print(f"⚠️ Warning: {len(missing_entities)} triples have missing entity/relation mappings.")

In [ ]:
relation_counts = ds["relation_id"].value_counts().reset_index()

relation_counts.columns = ["relation_id", "count"]

relation_counts = relation_counts.sort_values(by="count", ascending=False)

mean_conn = relation_counts['count'].mean()
rels_to_keep = relation_counts[(relation_counts['count'] > mean_conn//10) & (relation_counts['count'] < mean_conn)]['relation_id'].tolist()

ds_parsed = ds[ds['relation_id'].isin(rels_to_keep)]

number_of_proj = 2
shared_tails = ds_parsed.groupby('tail_id')['head_id'].nunique()
shared_tails = shared_tails[shared_tails > (number_of_proj-1)]  # tails with more than number_of_proj heads

shared_tails = shared_tails.sample(frac=1, random_state=77)  # shufflle series

In [ ]:
queries = [] 
shared_tails_inter = []
results = []  

if shared_tails.empty:
    print("No shared tails found with relation_id = 5")
else:
    for shared_tail_id in tqdm(shared_tails.index, desc="Processing queries"):
        # Get the heads pointing to this shared tail
        heads = ds_parsed[ds_parsed['tail_id'] == shared_tail_id]['head_id'].unique()[:number_of_proj]
        if len(heads) < number_of_proj:
            continue  # need at least number_of_proj heads

        relations = ds_parsed[ds_parsed['tail_id'] == shared_tail_id]['relation_id'].values

        query = [[(heads[i], relations[i]) for i in range(number_of_proj)]]

        new_head_id = shared_tail_id
        new_edges = ds_parsed[(ds_parsed['head_id'] == new_head_id) & (ds_parsed['relation_id'] != 0)]

        if new_edges.empty:
            continue
        
        # Group tails by relations
        relation_dict = (
            new_edges.groupby('relation_id')['tail_id']
            .apply(list)
            .to_dict()
        )
        
        for rel, tails in relation_dict.items():
            if rel in rels_to_keep:
                queries.append(query+[rel])
                shared_tails_inter.append(shared_tail_id)
                results.append(tails)

In [ ]:
save_dict = {
    'queries': queries,
    'results': results
}

with open(f'/home/cc/phd/KGEmbeddings/queries/{DATA}/queries.pkl', 'wb') as f:
    pickle.dump(save_dict, f)